# Timing CPU and GPU inference

Load datasets for testing performance in inference

In [1]:
model_path = "Training_AdaptiveHP_acc=0.7426_ebops=1001_VU_DA_bitfile/model_Training_AdaptiveHP_acc=0.7426_ebops=1001.keras"

x_test_path = 'Data/x_test.npy'
y_test_path = 'Data/y_test.npy'

iterations = 10

In [ ]:
# Check if GPU is available
import tensorflow as tf
device_name = tf.test.gpu_device_name()
if not device_name:
  compute = 'CPU'
  timings_path = 'Timings/CPU'
  print('GPU device not found')
else: 
  compute = 'GPU'
  timings_path = 'Timings/GPU'
  print('Found GPU at: {}'.format(device_name))

2026-06-09 10:36:12.248762: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-09 10:36:12.624784: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


SystemError: GPU device not found

In [2]:
import os
import sys
import time
import numpy as np
# Load input from .npy file
x_test = np.load(x_test_path)
y_test = np.load(y_test_path)

In [3]:
def test_dut():
    start = time.perf_counter()
    y_dut = model.predict(x_test, verbose=0)
    end = time.perf_counter() - start

    return [y_dut, end]

In [4]:
def cal_accuracy(y_dut):
    y_pred = np.argmax(y_dut, axis=1)
    y_true = np.argmax(y_test, axis=1)
    return np.sum(y_pred == y_true) / len(y_true)

Run the actual inference

In [6]:
from keras.models import load_model
import hgq.layers
from hgq.utils import trace_minmax

model = load_model(model_path)
# Calibrate datalane in HGQ2-model since it has layers with WRAP
#trace_minmax(model, x_test, verbose=True)


In [ ]:

timestamp = time.strftime("%Y%m%d_%H%M%S")
nr_samples = x_test.shape[0]
timings = []
for i in range(iterations):    
    # do inference
    result = test_dut()
    y_dut = result[0] 
    timings.append(result[1])

    acc = cal_accuracy(y_dut)
    print(f"\nTime: {result[0]}, Acc: {acc}")

timing_results_path = f"{timings_path}timings_dataset-{nr_samples}_{iterations}iterations_{timestamp}.txt"
np.savetxt(timing_results_path,timings)

2026-06-09 09:10:21.915954: I external/local_xla/xla/service/service.cc:163] XLA service 0x7ca77c00e090 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-06-09 09:10:21.915982: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Host, Default Version
2026-06-09 09:10:22.066238: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1780989023.059532  448012 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-06-09 09:10:37.532367: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



accuracy of hardware inference: 0.7435530120481928


2026-06-09 09:10:58.524479: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928


2026-06-09 09:11:43.076744: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


...

2026-06-09 09:12:46.950943: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928

accuracy of hardware inference: 0.7435530120481928


2026-06-09 09:15:03.978328: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



accuracy of hardware inference: 0.7435530120481928


KeyboardInterrupt: 

[Tensorflow profiler](https://github.com/tensorflow/tensorboard/blob/master/docs/tensorboard_profiling_keras.ipynb) 

In [16]:
#!pip install -U tensorboard_plugin_profile

  Using cached etils-1.14.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━ 14.4/25.9 MB 1.1 MB/s eta 0:00:11
Resuming download xprof-2.22.1-cp312-none-manylinux_2_27_x86_64.whl (14.4 MB/25.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.9/25.9 MB 805.1 kB/s  0:00:14m0:00:0100:01
Using cached etils-1.14.0-py3-none-any.whl (172 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 952.9 kB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 954.6 kB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 1.1 MB/s  0:00:06 eta 0:00:01
  Attempting uninstall: grpciom━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/35 [oauthlib]
    Found existing installation: grpcio 1.78.0━━━━━━━━━━━━━━━━━━━━  8/35 [grpcio]
    Uninstalling grpcio-1.78.0:m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  8/35 [grpcio]
      Successfully uninstalled grpcio-1.78.0━━━━━━━━━━━━━━━━━━

In [7]:
import tensorflow as tf
tf.profiler.experimental.start("logs/profile")

model.predict(x_test, batch_size=256)

tf.profiler.experimental.stop()

# Load the TensorBoard notebook extension.
%load_ext tensorboard
# Launch TensorBoard and navigate to the Profile tab to view performance profile
%tensorboard --logdir=logs

2026-06-09 09:29:32.436999: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:103] Profiler session initializing.
2026-06-09 09:29:32.437045: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:118] Profiler session started.
2026-06-09 09:29:35.968820: I external/local_xla/xla/service/service.cc:163] XLA service 0x7d605401f940 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-06-09 09:29:35.968846: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Host, Default Version
2026-06-09 09:29:36.112601: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


  96/3243 ━━━━━━━━━━━━━━━━━━━━ 2:21 45ms/step 

I0000 00:00:1780990176.960159  464397 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


3232/3243 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

2026-06-09 09:29:39.737360: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


3243/3243 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step


2026-06-09 09:29:42.927826: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:68] Profiler session collecting data.
2026-06-09 09:29:46.544638: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:136] Profiler session tear down.
2026-06-09 09:29:46.550490: I external/local_xla/xla/tsl/profiler/rpc/client/save_profile.cc:150] Collecting XSpace to repository: logs/profile/plugins/profile/2026_06_09_09_29_46/krisslaptoppop.xplane.pb


Reusing TensorBoard on port 6006 (pid 458249), started 0:08:48 ago. (Use '!kill 458249' to kill it.)